# cycle-detection-temp-set — ex1: add temp-set cycle detection to a DFS traversal

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cycle-detection-temp-set`. Running the final beacon cell reports progress against the `Backprop: cycle detection via temp set` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: cycle detection via temp set` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cycle-detection-temp-set`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cycle-detection-temp-set"
DD_SUBTOPIC = "Backprop: cycle detection via temp set"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cycle detection via temp set — quick refresher

In a DFS topological sort, the **temp** (gray) set holds every node currently on the recursion stack. A cycle is exactly a back-edge: a node being visited that's already in temp.

```python
def visit(node):
    if id(node) in temp:
        raise ValueError(f'cycle through {node!r}')
    temp.add(id(node))
    for child in get_children(node):
        visit(child)
    temp.remove(id(node))
    perm.add(id(node))
```

Why a SEPARATE set from `perm`? `perm` is 'I have finished this subtree, skip it' (good — avoids re-traversal of shared descendants in a DAG). `temp` is 'I am currently inside this subtree' — meeting it again means we walked in a circle. Without the split, you can't tell 'shared descendant' from 'cycle'.

Use `id(node)` as the key — the input nodes might not be hashable (or might have value-equality that gives false positives).

### Exercise 1 — add temp-set cycle detection to a DFS traversal

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the temp-set (gray) back-edge detection pattern to distinguish 'already-visited shared descendant' (legal) from 'cycle' (illegal) inside a DFS traversal.
> Keywords: cycle-detection, temp-set, gray, back-edge
> ```

**KCs targeted:** `cycle-detection-temp-set`, `dfs-three-set-toposort`

Implement `has_cycle(root, get_children)` — return `True` if the directed graph reachable from `root` contains a cycle, `False` otherwise.

**Why this is non-trivial.** A diamond DAG (multiple paths to the same descendant) is NOT a cycle. Naive 'mark visited; raise if you see a visited node' gives false positives:

```
    a
   / \
  b   c
   \ /
    d            ← a, b, c, d, then we reach d again via c → false alarm
```

Correct algorithm uses **two** sets:
- `perm` — 'I have FINISHED processing this subtree'. Re-seeing a `perm` node is fine; it's a shared descendant in a DAG.
- `temp` — 'I am CURRENTLY inside this subtree (still on the recursion stack)'. Re-seeing a `temp` node IS a cycle.

```python
def visit(node):
    if id(node) in perm: return         # legal — already finished
    if id(node) in temp: return True    # CYCLE — back-edge
    temp.add(id(node))
    for child in get_children(node):
        if visit(child) is True:
            return True
    temp.remove(id(node))
    perm.add(id(node))
    return False
```

Return type: `bool`. Do NOT raise — the caller picks what to do with the result. (Compare with the sibling `dfs-three-set-toposort` atom, which DOES raise; same `temp` set, different response.)

Use `id(node)` as the set key.

In [ ]:
def has_cycle(root, get_children) -> bool:
    """Return True if reachable graph contains a cycle, False otherwise."""
    raise NotImplementedError()


def _test_ex1():
    # --- helper node ---
    class N:
        def __init__(self, name, *children):
            self.name = name
            self.children = list(children)
        def __repr__(self):
            return f'N({self.name})'

    def get_children(n):
        return n.children

    # --- acyclic linear chain → False ---
    c = N('c')
    b = N('b', c)
    a = N('a', b)
    assert has_cycle(a, get_children) is False, 'linear chain is acyclic'

    # --- diamond DAG → False (shared descendant is NOT a cycle) ---
    d = N('d')
    b = N('b', d)
    c = N('c', d)
    a = N('a', b, c)
    assert has_cycle(a, get_children) is False, (
        'diamond DAG is acyclic — shared descendants must use perm, not raise'
    )

    # --- two-node cycle → True ---
    x = N('x')
    y = N('y')
    x.children = [y]
    y.children = [x]
    assert has_cycle(x, get_children) is True, 'x → y → x is a cycle'

    # --- self-loop → True ---
    s = N('s')
    s.children = [s]
    assert has_cycle(s, get_children) is True, 's → s is a cycle'

    # --- cycle deeper in the graph → True ---
    # p → q → r → q (cycle between q and r)
    p = N('p')
    q = N('q')
    r = N('r')
    p.children = [q]
    q.children = [r]
    r.children = [q]
    assert has_cycle(p, get_children) is True, 'mid-graph cycle'

    # --- singleton (no children) → False ---
    lonely = N('lonely')
    assert has_cycle(lonely, get_children) is False

    # --- complex DAG with multiple diamonds → False ---
    leaf = N('leaf')
    m1 = N('m1', leaf)
    m2 = N('m2', leaf)
    n1 = N('n1', m1, m2)
    n2 = N('n2', m1, m2)
    root = N('root', n1, n2)
    assert has_cycle(root, get_children) is False, (
        'nested diamonds with shared descendants must NOT report a cycle'
    )

    # --- branching graph where ONE branch has a cycle → True ---
    # root -> good (acyclic chain)
    #      -> bad (cycle)
    good_c = N('good_c')
    good_b = N('good_b', good_c)
    good_a = N('good_a', good_b)
    bad_x = N('bad_x')
    bad_y = N('bad_y')
    bad_x.children = [bad_y]
    bad_y.children = [bad_x]
    root = N('root', good_a, bad_x)
    assert has_cycle(root, get_children) is True, (
        'cycle in a subgraph must propagate up to True'
    )

    # --- temp set MUST be popped on return (otherwise sibling subtrees ---
    # --- false-positive as cycles)                                     ---
    # Graph: root → x → leaf
    #             → y → leaf      ← x and y both child of root; both reach leaf
    # If you forget to remove from temp after a subtree finishes,
    # the second visit to leaf would (wrongly) hit temp and report a cycle.
    leaf = N('leaf')
    x = N('x', leaf)
    y = N('y', leaf)
    root = N('root', x, y)
    assert has_cycle(root, get_children) is False, (
        'two siblings sharing a leaf is NOT a cycle — '
        'did you forget to temp.remove() after each subtree?'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def has_cycle(root, get_children) -> bool:
    perm = set()   # fully processed subtree (NOT a cycle)
    temp = set()   # currently on the DFS stack (BACK-EDGE means cycle)

    def visit(node) -> bool:
        nid = id(node)
        if nid in perm:
            return False   # already finished — legal shared descendant
        if nid in temp:
            return True    # back-edge — cycle
        temp.add(nid)
        for child in get_children(node):
            if visit(child):
                return True
        temp.remove(nid)   # MUST remove on the way out — else siblings false-positive
        perm.add(nid)
        return False

    return visit(root)
```

**Two color sets, two purposes.** One color (visited / not) would catch any re-visit — but that flags diamond DAGs as cycles. The split between `temp` (currently in flight) and `perm` (already finished) is the minimum information needed to distinguish 'shared descendant' from 'back-edge'.

**Why `temp.remove(nid)` matters.** When a subtree finishes, its root leaves the recursion stack — so its `id` must leave the `temp` set too. Otherwise, two sibling subtrees that share a leaf will report a false cycle: the first subtree adds the leaf's id to `temp`, finishes without removing it, then the second subtree tries to visit the leaf and sees it in `temp` → false alarm. The dedicated test at the bottom catches this.

**Vs. the sibling `dfs-three-set-toposort` atom.** Same `temp` machinery; that atom RAISES on the back-edge, this one RETURNS `True`. Different ergonomics for different callers — but the cycle-detection logic itself is the shared insight.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()